In [5]:
import pandas as pd
import numpy as np
import networkx as nx
import time
from algorithms import spnsa, _single_source_shortest_path_basic, calculate_reliance_from_sources, draw_graph
import random
from typing import Set

In [6]:
import importlib
import algorithms              # make sure the module itself is imported
importlib.reload(algorithms)   # reloads algorithms.py from disk

from algorithms import spnsa, _single_source_shortest_path_basic, calculate_reliance_from_sources, draw_graph

In [10]:




def create_graph(edgelist_csv: str, directed: bool = True) -> nx.Graph:
    """
    Build a NetworkX graph from Elliptic edgelist CSV using ORIGINAL tx ids.
    Expected columns: txId1, txId2
    """
    df = pd.read_csv(edgelist_csv, usecols=["txId1", "txId2"], dtype=str)
    df["txId1"] = df["txId1"].str.strip()
    df["txId2"] = df["txId2"].str.strip()

    G = nx.DiGraph() if directed else nx.Graph()
    G.add_edges_from(zip(df["txId1"].tolist(), df["txId2"].tolist()))
    return G


def get_illicit_nodes(classes_csv: str, illicit_label: str = "1") -> Set[str]:
    """
    Return illicit txIds (as ORIGINAL string ids).
    Expected columns: txId, class
    """
    df = pd.read_csv(classes_csv, usecols=["txId", "class"], dtype=str)
    df["txId"] = df["txId"].str.strip()
    df["class"] = df["class"].str.strip()

    return set(df.loc[df["class"] == str(illicit_label), "txId"].tolist())


def create_criminal_graph(classes_csv: str, edgelist_csv: str, illicit_label: str = "1",
                          directed: bool = True) -> nx.Graph:
    """
    Induced subgraph on illicit nodes: keep ONLY illicit nodes and edges between them.
    All node ids are ORIGINAL txIds (strings). No relabeling.
    """
    illicit_ids = get_illicit_nodes(classes_csv, illicit_label=illicit_label)

    e = pd.read_csv(edgelist_csv, usecols=["txId1", "txId2"], dtype=str)
    e["txId1"] = e["txId1"].str.strip()
    e["txId2"] = e["txId2"].str.strip()

    e_f = e[e["txId1"].isin(illicit_ids) & e["txId2"].isin(illicit_ids)]

    G = nx.DiGraph() if directed else nx.Graph()
    G.add_nodes_from(illicit_ids)  # include isolated illicit nodes too
    G.add_edges_from(zip(e_f["txId1"].tolist(), e_f["txId2"].tolist()))

    # optional label attribute
    nx.set_node_attributes(G, {tid: 1 for tid in illicit_ids}, "y")
    return G


def find_largest_component(
    G: nx.Graph,
    component_type: str = "weak",  # used only if G is directed
) -> nx.Graph:
    """
    Return the largest component as an induced subgraph (copied).

    - If G is undirected: uses connected_components.
    - If G is directed: uses weakly_connected_components by default,
      or strongly_connected_components if component_type="strong".
    """
    if G.number_of_nodes() == 0:
        # preserve graph type
        return G.copy()

    if G.is_directed():
        if component_type not in {"weak", "strong"}:
            raise ValueError("component_type must be 'weak' or 'strong' for directed graphs.")
        comps = (
            nx.weakly_connected_components(G)
            if component_type == "weak"
            else nx.strongly_connected_components(G)
        )
    else:
        comps = nx.connected_components(G)

    largest_cc = max(comps, key=len)
    return G.subgraph(largest_cc).copy()

def get_connected_components(G):
    if nx.number_connected_components(G) == 1:
        return [G]
        
    connected_components_generatorset = nx.connected_components(G)
    connected_components = []
    for component in connected_components_generatorset:
        component_graph = G.subgraph(component).copy()
        connected_components.append(component_graph)
    return connected_components

def get_eccentricity_distribution_list(component_list):
    ''' component_list : List[component] '''
    eccentricity_distribution_list = []
    for component in component_list:
        eccentricity_distribution = nx.eccentricity(component)
        eccentricity_distribution_list.append(eccentricity_distribution)
    return eccentricity_distribution_list

In [18]:
G_all = create_graph("../../datasets/elliptic/elliptic_txs_edgelist.csv", directed=True)
print(G_all)

DiGraph with 203769 nodes and 234355 edges


In [19]:
G_criminal = create_criminal_graph(
    classes_csv="../../datasets/elliptic/elliptic_txs_classes.csv",
    edgelist_csv="../../datasets/elliptic/elliptic_txs_edgelist.csv",
    illicit_label="1",
    directed=True
)

print(G_criminal)


DiGraph with 4545 nodes and 998 edges


In [20]:
criminal_nodes = get_suspicious_nodes("../../datasets/elliptic/elliptic_txs_classes.csv")
criminal_nodes_list = list(criminal_nodes)
all_nodes_list = list(G_all.nodes())

In [21]:
print(len(criminal_nodes))


4545


In [22]:
upto = 10
out_degree = dict(G_all.out_degree())
out_degree_largest = sorted(out_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
out_degree_largest = dict(out_degree_largest)
out_degree_largest

{'2984918': 472,
 '89273': 288,
 '102570': 122,
 '3181': 112,
 '7952': 99,
 '1891081': 95,
 '143705': 92,
 '565334': 90,
 '488266': 88,
 '793584': 82}

In [23]:
upto = 10
in_degree = dict(G_all.in_degree())
in_degree_largest = sorted(in_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
in_degree_largest = dict(in_degree_largest)
in_degree_largest

{'43388675': 284,
 '68705820': 247,
 '30699343': 241,
 '96576418': 239,
 '225859042': 212,
 '279187194': 211,
 '234890810': 199,
 '196107869': 188,
 '43397277': 182,
 '68706499': 178}

In [24]:
upto = 10

degree_diff = {
    node: abs(in_degree.get(node, 0) - out_degree.get(node, 0))
    for node in G_all.nodes()
}

degree_diff_largest = sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:upto]
degree_diff_largest = dict(degree_diff_largest)
degree_diff_largest

{'2984918': 471,
 '89273': 287,
 '43388675': 284,
 '68705820': 247,
 '30699343': 241,
 '96576418': 239,
 '225859042': 212,
 '279187194': 211,
 '234890810': 199,
 '196107869': 188}

In [25]:
len(degree_diff_largest.keys() & in_degree_largest.keys())

8

# Ex0: Uniformly random feed experiments

In [26]:
# feed size 200; radius 1
number_of_suspicious_nodes = []
number_of_nodes = []
number_of_edges = []
for i in range(100):
    feed = random.sample(all_nodes_list, 200)
    SPN, paths = spnsa(G_all, feed, radius=1)
    num_suspicious_nodes = len(SPN.nodes() & criminal_nodes)
    number_of_suspicious_nodes.append(num_suspicious_nodes)
    number_of_nodes.append(SPN.number_of_nodes())
    number_of_edges.append(SPN.number_of_edges())
    
avg_num_nodes = sum(number_of_nodes) / len(number_of_nodes)
avg_num_edges = sum(number_of_edges) / len(number_of_edges)
avg_num_suspicious_nodes = sum(number_of_suspicious_nodes) / len(number_of_suspicious_nodes)

print(f'DiGraph with average {avg_num_nodes} and {avg_num_edges} edges')
print(f'Number of suspicious nodes in SPN: {avg_num_suspicious_nodes}')
print(f'Criminal ration in SPN: {avg_num_suspicious_nodes / avg_num_nodes}')

DiGraph with average 364.77 and 165.95 edges
Number of suspicious nodes in SPN: 8.57
Criminal ration in SPN: 0.023494256654878417


In [27]:
# feed size 200; radius 2
number_of_suspicious_nodes = []
number_of_nodes = []
number_of_edges = []

for _ in range(10):
    feed = random.sample(all_nodes_list, 200)
    SPN, paths = spnsa(G_all, feed, radius=2)
    num_suspicious_nodes = len(SPN.nodes() & criminal_nodes)
    number_of_suspicious_nodes.append(num_suspicious_nodes)
    number_of_nodes.append(SPN.number_of_nodes())
    number_of_edges.append(SPN.number_of_edges())
    
avg_num_nodes = sum(number_of_nodes) / len(number_of_nodes)
avg_num_edges = sum(number_of_edges) / len(number_of_edges)
avg_num_suspicious_nodes = sum(number_of_suspicious_nodes) / len(number_of_suspicious_nodes)

print(f'DiGraph with average {avg_num_nodes} and {avg_num_edges} edges')
print(f'Number of suspicious nodes in SPN: {avg_num_suspicious_nodes}')
print(f'Criminal ration in SPN: {avg_num_suspicious_nodes / avg_num_nodes}')

DiGraph with average 497.8 and 300.9 edges
Number of suspicious nodes in SPN: 10.3
Criminal ration in SPN: 0.0206910405785456


In [28]:
# feed size 500; radius 1
number_of_suspicious_nodes = []
number_of_nodes = []
number_of_edges = []

for _ in range(100):
    feed = random.sample(all_nodes_list, 500)
    SPN, paths = spnsa(G_all, feed, radius=1)
    num_suspicious_nodes = len(SPN.nodes() & criminal_nodes)
    number_of_suspicious_nodes.append(num_suspicious_nodes)
    number_of_nodes.append(SPN.number_of_nodes())
    number_of_edges.append(SPN.number_of_edges())
    
avg_num_nodes = sum(number_of_nodes) / len(number_of_nodes)
avg_num_edges = sum(number_of_edges) / len(number_of_edges)
avg_num_suspicious_nodes = sum(number_of_suspicious_nodes) / len(number_of_suspicious_nodes)

print(f'DiGraph with average {avg_num_nodes} and {avg_num_edges} edges')
print(f'Number of suspicious nodes in SPN: {avg_num_suspicious_nodes}')
print(f'Criminal ration in SPN: {avg_num_suspicious_nodes / avg_num_nodes}')

DiGraph with average 907.0 and 413.81 edges
Number of suspicious nodes in SPN: 21.73
Criminal ration in SPN: 0.02395810363836825


# Ex1: Largest degree difference feed experiments

In [29]:
# feed size 200; radius 1
feed = dict(sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(G_all, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 293 nodes and 140 edges
Number of suspicious nodes in SPN: 12
Criminal ration in SPN: 0.040955631399317405


In [30]:
# feed size 200; radius 2
feed = dict(sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(G_all, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 351 nodes and 232 edges
Number of suspicious nodes in SPN: 12
Criminal ration in SPN: 0.03418803418803419


In [31]:
# feed size 500; radius 1
feed = dict(sorted(degree_diff.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(G_all, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 668 nodes and 372 edges
Number of suspicious nodes in SPN: 24
Criminal ration in SPN: 0.03592814371257485


# Ex2: Non-zero in-degree & zero out-degree experiments (Collect)

In [32]:
upto = 10

nonzero_in_degree = {
    node: in_degree.get(node, 0)
    for node in G_all.nodes() if out_degree.get(node, 0) == 0
}

nonzero_in_degree_largest = sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
nonzero_in_degree_largest = dict(nonzero_in_degree_largest)
nonzero_in_degree_largest

{'43388675': 284,
 '68705820': 247,
 '30699343': 241,
 '96576418': 239,
 '225859042': 212,
 '279187194': 211,
 '234890810': 199,
 '196107869': 188,
 '43397277': 182,
 '68706499': 178}

In [33]:
# feed size 200; radius 1
feed = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(G_all, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 200 nodes and 0 edges
Number of suspicious nodes in SPN: 21
Criminal ration in SPN: 0.105


In [34]:
# feed size 200; radius 2
feed = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(G_all, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 200 nodes and 0 edges
Number of suspicious nodes in SPN: 21
Criminal ration in SPN: 0.105


In [35]:
# feed size 500; radius 1
feed = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(G_all, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 500 nodes and 0 edges
Number of suspicious nodes in SPN: 24
Criminal ration in SPN: 0.048


# Ex3: Non-zero out-degree & zero in-degree experiments (Distribute)

In [36]:
upto = 10

nonzero_out_degree = {
    node: out_degree.get(node, 0)
    for node in G_all.nodes() if in_degree.get(node, 0) == 0
}

nonzero_out_degree_largest = sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
nonzero_out_degree_largest = dict(nonzero_out_degree_largest)
nonzero_out_degree_largest

{'102570': 122,
 '1891081': 95,
 '143705': 92,
 '4600600': 76,
 '3185686': 72,
 '1192279': 69,
 '5678249': 67,
 '847323': 64,
 '65253': 53,
 '3180883': 51}

In [37]:
# feed size 200; radius 1
feed = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(G_all, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 435 nodes and 274 edges
Number of suspicious nodes in SPN: 0
Criminal ration in SPN: 0.0


In [38]:
# feed size 200; radius 2
feed = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(G_all, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 586 nodes and 472 edges
Number of suspicious nodes in SPN: 5
Criminal ration in SPN: 0.008532423208191127


In [39]:
# feed size 500; radius 1
feed = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(G_all, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 1035 nodes and 630 edges
Number of suspicious nodes in SPN: 9
Criminal ration in SPN: 0.008695652173913044


# Ex4: 50% Collect and 50% Distribute feed experiments

In [40]:
# feed size 200; radius 1
upto = 200
nonzero_in_degree_largest = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
nonzero_out_degree_largest = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
feed = nonzero_in_degree_largest | nonzero_out_degree_largest

SPN, paths = spnsa(G_all, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')


DiGraph with 316 nodes and 157 edges
Number of suspicious nodes in SPN: 11
Criminal ration in SPN: 0.03481012658227848


In [41]:
# feed size 200; radius 2
upto = 200
nonzero_in_degree_largest = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
nonzero_out_degree_largest = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
feed = nonzero_in_degree_largest | nonzero_out_degree_largest

SPN, paths = spnsa(G_all, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')


DiGraph with 411 nodes and 296 edges
Number of suspicious nodes in SPN: 12
Criminal ration in SPN: 0.029197080291970802


In [42]:
# feed size 500; radius 1
upto = 500
nonzero_in_degree_largest = dict(sorted(nonzero_in_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
nonzero_out_degree_largest = dict(sorted(nonzero_out_degree.items(), key=lambda x: x[1], reverse=True)[:upto//2])
feed = nonzero_in_degree_largest | nonzero_out_degree_largest

SPN, paths = spnsa(G_all, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')


DiGraph with 774 nodes and 454 edges
Number of suspicious nodes in SPN: 24
Criminal ration in SPN: 0.031007751937984496


# Ex5: High illicit degrees as feed experiments

In [43]:
upto = 10
illicit_degree = dict(G_criminal.degree())
illicit_degree_largest = sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:upto]
illicit_degree_largest = dict(illicit_degree_largest)
illicit_degree_largest

{'13735016': 10,
 '99636452': 5,
 '209952974': 5,
 '100045534': 5,
 '85008951': 5,
 '219565918': 4,
 '96581586': 4,
 '218398211': 4,
 '209692891': 4,
 '115333920': 4}

In [44]:
# feed size 200; radius 1
feed = dict(sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(G_all, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 334 nodes and 191 edges
Number of suspicious nodes in SPN: 322
Criminal ration in SPN: 0.9640718562874252


In [45]:
# feed size 200; radius 2
feed = dict(sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:200])
SPN, paths = spnsa(G_all, feed, radius=2)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 416 nodes and 309 edges
Number of suspicious nodes in SPN: 386
Criminal ration in SPN: 0.9278846153846154


In [46]:
# feed size 500; radius 1
feed = dict(sorted(illicit_degree.items(), key=lambda x: x[1], reverse=True)[:500])
SPN, paths = spnsa(G_all, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 625 nodes and 498 edges
Number of suspicious nodes in SPN: 592
Criminal ration in SPN: 0.9472


In [47]:
# random 200 feed; radius 1
import random

feed = dict(random.sample(list(illicit_degree.items()), 200))
SPN, paths = spnsa(G_all, feed, radius=1)
suspicious_nodes_SPN = len(SPN.nodes() & criminal_nodes)
print(SPN)
print(f'Number of suspicious nodes in SPN: {suspicious_nodes_SPN}')
print(f'Criminal ration in SPN: {suspicious_nodes_SPN / SPN.number_of_nodes()}')

DiGraph with 304 nodes and 117 edges
Number of suspicious nodes in SPN: 246
Criminal ration in SPN: 0.8092105263157895
